In [1]:
import re
from pathlib import Path
import pandas as pd

PDF_PATH = Path("Report_DCA25MA108_Combined Transcript - FINAL-Rel.pdf")  # change if needed

# Time when the airplane CVR joins the combined transcript (from the report).
# Keep it as a parameter so you can adjust for other reports.
AIRPLANE_START_TIME = "20:43:06.2"

In [3]:
%pip install -q pdfplumber

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\huawei\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
# extract all PDF text (requires: pip install pdfplumber)
import pdfplumber

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"

pages_text = []
with pdfplumber.open(str(PDF_PATH)) as pdf:
    for page in pdf.pages:
        pages_text.append(page.extract_text() or "")

full_text = "\n\n".join(pages_text)
print("Pages:", len(pages_text), "Chars:", len(full_text))

Pages: 34 Chars: 61163


In [5]:
# isolate the combined transcript section (start at the first timestamp near START OF COMBINED TRANSCRIPT)
# This makes parsing more stable and avoids front-matter.
start_marker = "START OF COMBINED TRANSCRIPT"
idx = full_text.find(start_marker)
if idx == -1:
    raise ValueError("Could not find 'START OF COMBINED TRANSCRIPT' in the PDF text.")

# Find the nearest timestamp before the marker, then slice from there
time_token_re = re.compile(r"\b\d{2}:\d{2}:\d{2}\.\d\b")
m = None
for mm in time_token_re.finditer(full_text, 0, idx):
    m = mm  # keep last match before marker

if not m:
    raise ValueError("Could not find a timestamp before START OF COMBINED TRANSCRIPT.")

transcript_text = full_text[m.start():]
print("Transcript slice chars:", len(transcript_text))

Transcript slice chars: 49178


In [6]:
# remove obvious page headers/footers lines (light cleanup)
lines = [ln.rstrip() for ln in transcript_text.splitlines()]

def is_header_line(ln: str) -> bool:
    s = ln.strip()
    if not s:
        return False

    # Common report header patterns
    if "COCKPIT VOICE RECORDERS AND AIR TRAFFIC CONTROL COMBINED TRANSCRIPT" in s:
        return True
    if s.startswith("<CHOOSE TYPE REPORT>"):
        return True
    if re.match(r"PG \d+ OF \d+", s):
        return True
    if s.startswith("DCA25MA108"):
        return True

    # Column headers that repeat
    if s.startswith("Time and") or s == "Source":
        return True
    if "PAT-25 (Helicopter)" in s or "DCA Tower Recording" in s or "JIA5342 (Airplane)" in s:
        return True

    return False

clean_lines = [ln for ln in lines if not is_header_line(ln)]
clean_text = "\n".join(clean_lines)
print("Clean text chars:", len(clean_text))

Clean text chars: 43917


In [7]:
# timestamp segmentation (robust even if PDF columns get merged into one line)
time_token_re = re.compile(r"\b\d{2}:\d{2}:\d{2}\.\d\b")
matches = list(time_token_re.finditer(clean_text))
print("Timestamp tokens found:", len(matches))

segments = []
for i, mm in enumerate(matches):
    t = mm.group(0)
    start = mm.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(clean_text)
    content = clean_text[start:end].strip()
    content = re.sub(r"\s+", " ", content).strip()
    if content:
        segments.append((t, content))

print("Segments:", len(segments))
print("First 3 segments:\n", segments[:3])

Timestamp tokens found: 559
Segments: 378
First 3 segments:
 [('20:30:18.0', 'START OF COMBINED TRANSCRIPT'), ('20:30:18.0', 'DCA-LC thank you Bluestreak fifty six seventy three winds are three two zero at one six gusts two five traffic on runway one will hold short of your intersection runway three three cleared for takeoff.'), ('20:30:19.1', 'RDO-2 Washington Tower PAT two five U-H sixty off of Montgomery requesting flight following back to Davison.')]


In [8]:
# helpers: time conversion + content cleanup + speaker classification
def time_to_seconds(t: str) -> float:
    hh, mm, rest = t.split(":")
    return int(hh) * 3600 + int(mm) * 60 + float(rest)

AIRPLANE_START_SEC = time_to_seconds(AIRPLANE_START_TIME)

def clean_content(s: str) -> str:
    # Remove repeated "Source Source Source" artifacts, collapse whitespace
    s = re.sub(r"\bSource\b(?:\s+Source\b)+", "", s, flags=re.IGNORECASE).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def classify(text: str, t: str) -> str:
    s = text.strip()
    u = s.upper()

    # ATC cues
    if u.startswith("DCA-LC") or u.startswith("TWR-A") or u.startswith("TWR-A2"):
        return "atc"

    # Helicopter CVR cues
    if u.startswith("INT-") or u.startswith("FPS") or u.startswith("PAT25"):
        return "helicopter"

    # Airplane CVR cues
    if u.startswith("HOT-") or u.startswith("CAM") or u.startswith("EGPWS") or u.startswith("TCAS") or u.startswith("JIA5342"):
        return "airplane"

    # RDO is ambiguous, decide based on time and keywords
    if u.startswith("RDO"):
        ts = time_to_seconds(t)
        if "PAT" in u or "U-H" in u or "UH" in u:
            return "helicopter"
        # after airplane joins, default RDO to airplane
        return "helicopter" if ts < AIRPLANE_START_SEC else "airplane"

    # Callsign heuristics
    if "PAT TWO FIVE" in u or u.startswith("PAT "):
        return "helicopter"
    if "BLUESTREAK" in u or "JIA5342" in u:
        return "airplane"

    # Default bucket
    return "atc"

In [9]:
# build the final dataframe in the requested format (time, helicopter, atc, airplane)
rows = []
for t, txt in segments:
    txt = clean_content(txt)
    if not txt:
        continue

    cat = classify(txt, t)
    rows.append({
        "time": t,
        "helicopter": txt if cat == "helicopter" else "",
        "atc": txt if cat == "atc" else "",
        "airplane": txt if cat == "airplane" else "",
    })

df = pd.DataFrame(rows)
print("Rows:", len(df))
df.head(10)

Rows: 378


,time,helicopter,atc,airplane
0,20:30:18.0,,START OF COMBINED TRANSCRIPT,
1,20:30:18.0,,DCA-LC thank you Bluestreak fifty six seventy ...,
2,20:30:19.1,RDO-2 Washington Tower PAT two five U-H sixty ...,,
3,20:30:27.2,INT-2 wow not flight following *.,,
4,20:30:27.6,,,JIA5673 cleared for takeoff runway three three...
5,20:30:28.1,INT-1 ha.,,
6,20:30:33.7,INT-2 hopefully they didn't hear that but they...,,
7,20:30:41.5,INT-1 they just said uh we'll ignore that.,,
8,20:30:43.7,INT-2 yeah. I don't hear any traffic on there ...,,
9,20:30:48.6,,TWR-A * in sight departing runway three three ...,


In [10]:
# export CSV
OUT_CSV = Path("combined_transcript_timeline.csv")
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV.resolve())

Saved: C:\32Madjda\Academic\UdeM\Art\AlgoritmicArt\CVR\combined_transcript_timeline.csv


In [11]:
# quick sanity checks around the collision window, adjust times if you want
def in_window(t, start, end):
    s = time_to_seconds(t)
    return time_to_seconds(start) <= s <= time_to_seconds(end)

collision = df[df["time"].apply(lambda x: in_window(x, "20:47:37.0", "20:48:00.1"))]
collision

,time,helicopter,atc,airplane
357,20:47:37.2,,,HOT-1 cool.
358,20:47:37.8,,DCA-LC [two brief mic keys with rapid beeping ...,
359,20:47:39.1,,TWR-A PAT two five you have the C-R-J in sight...,
360,20:47:40.3,,,TCAS traffic. traffic. [automated voice]
361,20:47:42.0,,TWR-A PAT [transmission interrupted by 0.8 sec...,
362,20:47:44.1,RDO-1 PAT two five has uh— aircraft in sight r...,,
363,20:47:47.3,,TWR-A *. DCA-LC vis separation. TWR-A vis sep ...,
364,20:47:47.8,INT-2 (woah/below).,,
365,20:47:52.5,INT-1 alright kinda come left for me ma'am I t...,,
366,20:47:54.3,INT-2 sure.,,
